jt -r

# Autocorrelación

## Cuestiones

In [1]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pylab as plt
from wooldridge import *

Usando los datos de **barium**, estima el modelo $log(chempi)= \beta_0 + \beta_1 log(gas) + \beta_2 log(rtwex) + \beta_3 befile6  + \beta_4 affile6 + \beta_5  afdec6 + u$, y analiza la posible existencia de autocorrelación en el modelo. En caso afirmativo, estima el modelo corregido.

In [2]:
dataWoo('barium', description=True)
datos=dataWoo('barium')
y=datos['lchempi']
X=datos[['lgas', 'befile6', 'affile6', 'afdec6']]
mco4=sm.OLS(y, sm.add_constant(X)).fit()
mco4.summary()

name of dataset: barium
no of variables: 31
no of observations: 131

+----------+---------------------------------+
| variable | label                           |
+----------+---------------------------------+
| chnimp   | Chinese imports, bar. chl.      |
| bchlimp  | total imports bar. chl.         |
| befile6  | =1 for all 6 mos before filing  |
| affile6  | =1 for all 6 mos after filing   |
| afdec6   | =1 for all 6 mos after decision |
| befile12 | =1 all 12 mos before filing     |
| affile12 | =1 all 12 mos after filing      |
| afdec12  | =1 all 12 mos after decision    |
| chempi   | chemical production index       |
| gas      | gasoline production             |
| rtwex    | exchange rate index             |
| spr      | =1 for spring months            |
| sum      | =1 for summer months            |
| fall     | =1 for fall months              |
| lchnimp  | log(chnimp)                     |
| lgas     | log(gas)                        |
| lrtwex   | log(rtwex)               

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                lchempi   R-squared:                       0.055
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     1.821
Date:                Mon, 04 Dec 2023   Prob (F-statistic):              0.129
Time:                        18:16:17   Log-Likelihood:                 103.71
No. Observations:                 131   AIC:                            -197.4
Df Residuals:                     126   BIC:                            -183.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.9695      3.511     -1.131      0.260     -10.918       2.979
lgas           0.3836      0.154      2.499      0.014       0.080       0.687
befile6       -0.0275      0.047     -0.585      0.559      -0.120       0.065
affile6        0.0223      0.047      0.469      0.640      -0.072       0.116
afdec6         0.0600      0.048      1.239      0.217      -0.036       0.156
==============================================================================
Omnibus:                        3.775   Durbin-Watson:                   0.066
Prob(Omnibus):                  0.151   Jarque-Bera (JB):                2.550
Skew:                           0.160   Prob(JB):                        0.279
Kurtosis:                       2.396   Cond. No.                     8.24e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.24e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [3]:
from statsmodels.stats.stattools import durbin_watson
dw=durbin_watson(mco4.resid)
dw

0.06619733488532661

In [5]:

rho= 1 - dw/2 # dw = 2(1-rho) => rho = 1 - DW/2
print(rho)
mco_autocorr=sm.GLSAR(y, sm.add_constant(X), rho=rho)
res=mco_autocorr.iterative_fit(maxiter=100,rtol=10**(-10))


print ('Iteraciones = %d Converge: %s' % (res.iter, res.converged) )
print ('Rho =  ', mco_autocorr.rho)
print(res.summary())

0.9669013325573367
Iteraciones = 5 Converge: True
Rho =   [0.96583123]
                           GLSAR Regression Results                           
Dep. Variable:                lchempi   R-squared:                       0.008
Model:                          GLSAR   Adj. R-squared:                 -0.024
Method:                 Least Squares   F-statistic:                    0.2590
Date:                Mon, 04 Dec 2023   Prob (F-statistic):              0.904
Time:                        18:17:00   Log-Likelihood:                 363.67
No. Observations:                 130   AIC:                            -717.3
Df Residuals:                     125   BIC:                            -703.0
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------

5. Usando los datos de **prminwge**, estima el modelo lineal $\log( prepopt) = \beta_0 + \beta_1 \log(mincovt) + \beta_2 \log(usgnpt) + u$, y analiza la posible autocorrelación del modelo con el contraste de Ljung-Box, para distintos retardos.

In [7]:
dataWoo('prminwge', description=True)

name of dataset: prminwge
no of variables: 25
no of observations: 38

+----------+---------------------------------+
| variable | label                           |
+----------+---------------------------------+
| year     | 1950-1987                       |
| avgmin   | weighted avg min wge, 44 indust |
| avgwage  | wghted avg hrly wge, 44 indust  |
| kaitz    | Kaitz min wage index            |
| avgcov   | wghted avg coverage, 8 indust   |
| covt     | economy-wide coverage of min wg |
| mfgwage  | avg manuf. wage                 |
| prdef    | Puerto Rican price deflator     |
| prepop   | PR employ/popul ratio           |
| prepopf  | PR employ/popul ratio, alter.   |
| prgnp    | PR GNP                          |
| prunemp  | PR unemployment rate            |
| usgnp    | US GNP                          |
| t        | time trend:  1 to 38            |
| post74   | time trend:  starts in 1974     |
| lprunemp | log(prunemp)                    |
| lprgnp   | log(prgnp)              

In [8]:
datos=dataWoo('prminwge')
y=datos['lprepop']
X=datos[['lmincov', 'lusgnp']]
        
mco5=sm.OLS(y, sm.add_constant(X)).fit()
mco5.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                lprepop   R-squared:                       0.660
Model:                            OLS   Adj. R-squared:                  0.641
Method:                 Least Squares   F-statistic:                     34.04
Date:                Mon, 04 Dec 2023   Prob (F-statistic):           6.17e-09
Time:                        18:17:18   Log-Likelihood:                 57.376
No. Observations:                  38   AIC:                            -108.8
Df Residuals:                      35   BIC:                            -103.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.0544      0.765     -1.378      0.177      -2.608       0.499
lmincov       -0.1544      0.065     -2.380      0.023      -0.286      -0.023
lusgnp        -0.0122      0.089     -0.138      0.891      -0.192       0.168
==============================================================================
Omnibus:                        0.079   Durbin-Watson:                   0.340
Prob(Omnibus):                  0.961   Jarque-Bera (JB):                0.084
Skew:                           0.073   Prob(JB):                        0.959
Kurtosis:                       2.822   Cond. No.                         676.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [9]:
from statsmodels.stats.diagnostic import acorr_ljungbox
acorr_ljungbox(mco5.resid, lags=3)

,lb_stat,lb_pvalue
1,27.943973,1.248793e-07
2,46.750919,7.049639e-11
3,57.880346,1.667117e-12


In [11]:
from statsmodels.stats.stattools import durbin_watson
dw=durbin_watson(mco5.resid)
print("DW: ", dw)

DW:  0.3396275701941493


In [12]:
rho= 1 - dw/2 # dw = 2(1-rho) => rho = 1 - DW/2
print(rho)
mco_autocorr=sm.GLSAR(y, sm.add_constant(X), rho=rho)
res=mco_autocorr.iterative_fit(maxiter=100,rtol=10**(-10))


print ('Iterations used = %d Converged %s' % (res.iter, res.converged) )
print ('Rho =  ', mco_autocorr.rho)
print(res.summary())

#acorr_ljungbox(mco_autocorr.resid, lags=1)

0.8301862149029253
Iterations used = 20 Converged True
Rho =   [0.92702613]
                           GLSAR Regression Results                           
Dep. Variable:                lprepop   R-squared:                       0.163
Model:                          GLSAR   Adj. R-squared:                  0.113
Method:                 Least Squares   F-statistic:                     3.301
Date:                Mon, 04 Dec 2023   Prob (F-statistic):             0.0490
Time:                        18:19:13   Log-Likelihood:                 79.333
No. Observations:                  37   AIC:                            -152.7
Df Residuals:                      34   BIC:                            -147.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------